# 📝 가설검정·회귀 과제 LV3 정답 — 뉴욕 택시 요금 구조 분석 (강사용)

각 단계의 **모범 코드 + 자가채점 + 해설** 입니다.

> 🔧 **이번 과제의 도구**: 검정은 이번 단원 주력인 **Pingouin**(`pg.normality`·`pg.homoscedasticity`·`pg.mwu`·`pg.kruskal`·`pg.chi2_independence`)으로, **회귀는 `statsmodels`**(`smf.ols`)로 갑니다. 전처리는 6일차에 배운 **pandas**(`to_datetime`·`dropna`·조건 필터·파생 열)를 그대로 씁니다.

In [ ]:
# [제공 코드] 전처리·검정·회귀에 쓸 라이브러리와 한글 폰트를 준비합니다.
import warnings
warnings.filterwarnings('ignore')   # 사소한 경고를 숨겨 출력을 깔끔하게
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
import scikit_posthocs as sp                          # 비모수 사후검정 Dunn (pingouin 에 없음)
from statsmodels.stats.contingency_tables import Table   # 카이제곱 사후분석 (조정된 잔차)
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={"axes.unicode_minus": False})

os.makedirs('output', exist_ok=True)          # 정제본을 저장할 폴더

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을 봅니다. (아래 셀은 실행만 하면 됩니다.)

**여기서 세 가지를 눈으로 확인하세요** — 1) `pickup`·`dropoff` 의 자료형이 `object`(문자열)라는 것, 2) 결측이 있는 열이 어디인지, 3) `distance`·`passengers` 의 **최솟값이 0** 이라는 것(거리 0마일·승객 0명인 운행은 정상적인 기록이 아닙니다).

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·요약
#   (미리보기 전용 변수 preview 를 씁니다. 문제 풀이용 df 는 문제 1 에서 직접 불러오세요.)
preview = pd.read_csv("../../day09_가설검정_회귀분석/data/taxis.csv")
print("행·열 크기:", preview.shape)
print("\n[앞 5행] head()"); display(preview.head())
print("\n[열·자료형·결측] info()"); preview.info()
print("\n[수치형 요약] describe()"); display(preview.describe().round(3))

---
## 1. 원본 데이터 전처리 — 분석할 수 있는 상태로 만들기

**배경**: 원본에는 1) 문자열로 저장된 날짜, 2) 다섯 개 열의 결측치, 3) 물리적으로 불가능한 값(거리 0마일·승객 0명·소요시간 0분)이 섞여 있습니다. 이걸 그대로 검정에 넣으면 결과가 왜곡됩니다. **하나의 `df` 를 이어서** 여섯 단계로 정제합니다.

| 단계 | 확인 항목 |
|---|---|
| 1단계 | 원본 `(6433, 14)`, 결측 있는 열 **5개**, 중복 행 **0개** |
| 2단계 | 날짜형 변환 + 파생 열 `소요시간_분` — 중앙값 **10.9** |
| 3단계 | 파생 열 `팁비율` (= `tip` ÷ `fare`) — 평균 **0.1692**, 열 수 **16** |
| 4단계 | 결측 행 제거 후 **6341** 행 |
| 5단계 | 이상치 행 제거 후 **6220** 행 |
| 6단계 | 정제본 저장 — `output/taxis_정제.csv` |

### 1단계 — 불러오기·구조 파악
**요구사항**:
- `../../day09_가설검정_회귀분석/data/taxis.csv` 를 `df` 로 불러오세요(**날짜 변환은 아직 하지 않습니다** — 2단계에서 합니다).
- 원본의 **행 수**를 `raw_rows`, **열 수**를 `raw_cols` 에 담으세요.
- **결측치가 하나라도 있는 열의 개수**를 `n_missing_cols` 에 담으세요(정수).
- **완전히 똑같이 중복된 행의 개수**를 `n_dup` 에 담으세요(정수).
- 위 네 값을 `print` 로 출력해 확인하세요.

**예시**
```
raw_rows        → 6433
raw_cols        → 14
n_missing_cols  → 5
n_dup           → 0
```

<details><summary>힌트</summary>

```text
접근방법:
- 결측 개수는 열별로 세어(isna 합계) 그 값이 0 보다 큰 열이 몇 개인지 센다.
- 중복 행은 duplicated 결과의 합이다.

세부구현:
1. read_csv 로 원본을 df 에 담는다
2. shape 에서 행 수와 열 수를 꺼낸다
3. 열별 결측 개수를 구해 0 보다 큰 것의 개수를 int 로 담는다
4. 중복 행 개수를 int 로 담는다
```

</details>

In [ ]:
df = pd.read_csv("../../day09_가설검정_회귀분석/data/taxis.csv")
raw_rows, raw_cols = df.shape
# isna().sum() 은 열별 결측 개수 -> > 0 으로 참/거짓을 만들고 다시 세면 '결측이 있는 열 수'가 된다.
n_missing_cols = int((df.isna().sum() > 0).sum())
n_dup = int(df.duplicated().sum())
print("원본 크기      :", raw_rows, "행 ×", raw_cols, "열")
print("결측 있는 열 수:", n_missing_cols)
print("중복 행 수     :", n_dup)
print("\n[열별 결측 개수]"); print(df.isna().sum()[df.isna().sum() > 0].to_string())

In [ ]:
# [자가채점]
assert raw_rows == 6433
assert raw_cols == 14
assert n_missing_cols == 5, '결측이 하나라도 있는 열이 몇 개인지 세세요'
assert n_dup == 0
print("✅ 1단계 통과!")

### 해설 — 문제 1 · 1단계
- **접근법**: 정제를 시작하기 전에 **무엇을 고쳐야 하는지 목록을 먼저 만드는** 단계입니다. 결측이 있는 열은 `payment`·`pickup_zone`·`dropoff_zone`·`pickup_borough`·`dropoff_borough` 다섯 개이고, 중복 행은 없습니다.
- **흔한 실수**: `df.isna().sum()` 은 **열별 결측 개수(Series)** 입니다. 여기에 다시 `.sum()` 을 걸면 **전체 결측 개수(186)** 가 나와 '열의 개수'와 달라집니다. `> 0` 으로 걸러 **열 수**를 세야 합니다.
- **더 생각해 볼 점**: 결측을 무조건 지우는 것이 정답은 아닙니다. 여기서는 `pickup_zone`·`dropoff_zone`(세부 지역)은 이번 분석에 쓰지 않으므로 **그 열의 결측 때문에 행을 버리지는 않습니다**(4단계에서 다시 다룹니다).

### 2단계 — 날짜형 변환과 파생 열 `소요시간_분`
**배경**: `pickup`·`dropoff` 는 **문자열**이라 그대로는 뺄셈이 안 됩니다. 날짜형으로 바꾼 뒤 두 시각의 차이로 **운행 소요시간**을 만듭니다. 소요시간은 뒤에서 회귀의 설명변수로 씁니다.

**요구사항**:
- `pd.to_datetime` 으로 `df['pickup']` 과 `df['dropoff']` 를 **날짜형으로 변환**하세요.
- 두 시각의 차이를 **분 단위 실수**로 만들어 `df['소요시간_분']` 열에 담으세요.
  (시간 차이는 `Timedelta` 라서 `.dt.total_seconds()` 로 초를 얻은 뒤 60 으로 나눕니다.)
- `소요시간_분` 의 **중앙값**을 `dur_median`, **최솟값**을 `dur_min` 에 담으세요.
- 중앙값·최솟값·최댓값을 `print` 로 출력하세요. **최솟값이 0** 이라는 점을 눈으로 확인하세요(5단계에서 처리합니다).

**예시**
```
round(dur_median, 2)  → 10.9
round(dur_min, 2)     → 0.0     # 0분짜리 운행 = 비정상 기록
```

<details><summary>힌트</summary>

```text
접근방법:
- 문자열 날짜를 날짜형으로 바꾸는 pandas 함수를 두 열에 각각 적용한다.
- 두 날짜형 열을 빼면 Timedelta 가 나온다. 초로 바꾼 뒤 60 으로 나누면 분이다.

세부구현:
1. pickup·dropoff 를 각각 날짜형으로 변환해 같은 열에 다시 담는다
2. (dropoff - pickup) 의 .dt.total_seconds() 를 60 으로 나눠 소요시간_분 열을 만든다
3. 그 열의 median·min 을 dur_median·dur_min 에 담고 max 와 함께 출력한다
```

</details>

In [ ]:
df["pickup"] = pd.to_datetime(df["pickup"])
df["dropoff"] = pd.to_datetime(df["dropoff"])
# 시각끼리 빼면 Timedelta 가 나온다 — 숫자로 쓰려면 total_seconds() 로 풀어 60 으로 나눈다.
df["소요시간_분"] = (df["dropoff"] - df["pickup"]).dt.total_seconds() / 60
dur_median = df["소요시간_분"].median()
dur_min = df["소요시간_분"].min()
print(f'소요시간(분) 중앙값 = {dur_median:.2f}')
print(f'소요시간(분) 최솟값 = {dur_min:.2f}  <- 0분 운행이 섞여 있다')
print(f'소요시간(분) 최댓값 = {df["소요시간_분"].max():.2f}')
print("0분 이하인 행 수    =", int((df["소요시간_분"] <= 0).sum()))

In [ ]:
# [자가채점]
assert str(df['pickup'].dtype).startswith('datetime'), 'pickup 을 날짜형으로 변환했는지 확인하세요'
assert str(df['dropoff'].dtype).startswith('datetime')
assert '소요시간_분' in df.columns
assert abs(dur_median - 10.9) < 0.01
assert abs(dur_min - 0.0) < 0.01
print("✅ 2단계 통과!")

### 해설 — 문제 1 · 2단계
- **접근법**: 날짜형으로 바꿔야 **뺄셈이 되고**, 뺀 결과인 `Timedelta` 는 `.dt.total_seconds()` 로 숫자가 됩니다. 이렇게 만든 `소요시간_분` 은 원본에 없던 정보라 **파생 변수**라 부릅니다.
- **흔한 실수**: `.dt.seconds` 는 **하루 미만의 나머지 초**만 돌려줍니다 — 이 데이터는 최장 운행이 108분이라 **우연히 같은 답이 나오지만**, 24시간을 넘는 간격이 하나라도 섞이면 그 행만 조용히 틀립니다. 누적 초가 필요하면 언제나 `.dt.total_seconds()` 를 쓰는 습관이 안전합니다.
- **더 생각해 볼 점**: 소요시간 최솟값이 0이라는 것은 **승·하차 시각이 같은 기록**이 있다는 뜻입니다. 취소된 호출이거나 미터기 오작동일 수 있는데, 원인을 알 수 없으니 분석에서 제외하는 편이 안전합니다(5단계).

### 3단계 — 파생 열 `팁비율`
**배경**: 팁 금액(`tip`)을 그대로 비교하면 **요금이 비싼 운행일수록 팁도 큰** 당연한 결과가 나옵니다. '얼마나 후하게 줬는가'를 보려면 **기본요금 대비 비율**로 바꿔야 공정합니다.

**요구사항**:
- `tip` 을 `fare` 로 나눈 값을 `df['팁비율']` 열에 담으세요.
- `팁비율` 의 **평균**을 `tip_rate_mean` 에, 이 시점의 **열 개수**를 `n_cols_after` 에 담으세요.
- 평균·최댓값과 열 개수를 `print` 로 출력하세요.

**예시**
```
round(tip_rate_mean, 4) → 0.1692
n_cols_after            → 16      # 원래 14 + 소요시간_분 + 팁비율
```

<details><summary>힌트</summary>

```text
접근방법:
- 두 열끼리 나누면 행마다 계산된 새 Series 가 나온다. 그것을 새 열에 담는다.

세부구현:
1. tip 열을 fare 열로 나눠 팁비율 열을 만든다
2. 팁비율의 mean 을 tip_rate_mean 에 담는다
3. df.shape 의 열 개수를 n_cols_after 에 담고 함께 출력한다
```

</details>

In [ ]:
# 팁 금액 그대로는 요금이 큰 운행이 무조건 커 보인다 — 요금으로 나눠 비율로 만들어야 비교가 된다.
df["팁비율"] = df["tip"] / df["fare"]
tip_rate_mean = df["팁비율"].mean()
n_cols_after = df.shape[1]
print(f'팁비율 평균 = {tip_rate_mean:.4f}')
print(f'팁비율 최대 = {df["팁비율"].max():.4f}')
print("열 개수     =", n_cols_after, "(원래 14 + 소요시간_분 + 팁비율)")

In [ ]:
# [자가채점]
assert '팁비율' in df.columns
assert abs(tip_rate_mean - 0.1692) < 0.001, '팁비율 = tip ÷ fare 입니다'
assert n_cols_after == 16
print("✅ 3단계 통과!")

### 해설 — 문제 1 · 3단계
- **접근법**: **비율로 정규화**하면 금액 규모가 다른 운행을 같은 잣대로 비교할 수 있습니다. `fare` 는 모든 행에서 0보다 커서 0으로 나누는 문제는 생기지 않습니다.
- **흔한 실수**: `tip / total` 로 나누면 통행료(`tolls`)와 팁 자신이 분모에 섞여 의미가 흐려집니다. '기본요금 대비 팁'이므로 분모는 `fare` 입니다.
- **더 생각해 볼 점**: 평균 팁비율 0.169 는 '평균적으로 17% 를 준다'로 읽히지만, 뒤에서 보듯 **현금 결제가 전부 0으로 섞여 있어** 이 평균 자체가 오해를 부릅니다. 요약값 하나만 보고 결론 내리면 안 되는 이유입니다(문제 2).

### 4단계 — 결측 행 제거
**배경**: 결측 행을 버릴 때는 **어느 열을 기준으로 볼지 먼저 정하는 것**이 원칙입니다. 이번 분석에 실제로 쓸 열은 `payment`·`pickup_borough`·`dropoff_borough` 세 개이므로, 그 세 열만 기준으로 삼습니다.

**요구사항**:
- `payment`·`pickup_borough`·`dropoff_borough` **세 열 기준으로만** 결측 행을 제거해 `df` 에 다시 담으세요.
- 제거 후 남은 행 수를 `rows_after_na`, **제거된 행 수**를 `removed_na` 에 담으세요.
- 두 값을 `print` 로 출력하세요.

**예시**
```
rows_after_na → 6341
removed_na    → 92
```

<details><summary>힌트</summary>

```text
접근방법:
- 결측 행 제거 함수에 '어느 열을 기준으로 볼지' 를 지정하는 인자가 있다. 세 열만 넘긴다.
- 제거된 행 수는 (제거 전 행 수 − 제거 후 행 수) 로 구한다.

세부구현:
1. 제거 전 행 수를 미리 변수에 담아 둔다
2. payment·pickup_borough·dropoff_borough 세 열을 기준으로 결측 행을 없애 df 에 다시 담는다
3. 남은 행 수와 (전 − 후) 를 각각 rows_after_na, removed_na 에 담아 출력한다
```

</details>

In [ ]:
before = df.shape[0]
# subset 을 주면 그 열들에 결측이 있는 행만 지운다 — 지정하지 않으면 한 칸만 비어도 행이 통째로 날아간다.
df = df.dropna(subset=["payment", "pickup_borough", "dropoff_borough"])
rows_after_na = df.shape[0]
removed_na = before - rows_after_na
print("결측 제거 후 행 수 :", rows_after_na)
print("제거된 행 수       :", removed_na)
left = df.isna().sum()
left = left[left > 0]
print("\n남은 결측:", "없음" if left.empty else "")
if not left.empty:
    print(left.to_string())

In [ ]:
# [자가채점]
assert rows_after_na == 6341, 'payment·pickup_borough·dropoff_borough 세 열만 기준으로 제거하세요'
assert removed_na == 92
assert df['payment'].isna().sum() == 0
print("✅ 4단계 통과!")

### 해설 — 문제 1 · 4단계
- **접근법**: `dropna(subset=[...])` 로 **기준 열을 명시하는 것**이 핵심입니다. 92행(1.4%)을 잃고 6341행이 남습니다.
- **흔한 실수**: **이 데이터에서는 `subset` 없이 `df.dropna()` 를 불러도 결과가 같습니다(6341행).** `borough`(자치구)는 `zone`(세부 지역)에서 유도된 값이라 **둘의 결측이 정확히 같은 행**에서 나기 때문입니다 — 우연히 일치한 것이지 일반적인 일이 아닙니다. 예를 들어 이 데이터에 결측이 많은 `tolls` 열이 하나 더 있었다면, `subset` 없는 `dropna()` 는 **쓰지도 않을 그 열 때문에** 행을 대거 버렸을 것입니다. 그래서 결과가 같더라도 **기준 열을 적어 두는 습관**이 맞습니다 — 나중에 열이 추가돼도 코드가 흔들리지 않습니다.
- **더 생각해 볼 점**: 결측을 지우는 대신 **채우는(imputation)** 선택도 있습니다. 다만 `payment`·`borough` 는 범주형이라 임의로 채우면 없는 사실을 지어내는 셈이고, 잃는 비율이 1.4%(92/6433)로 작아 제거가 무난합니다.

### 5단계 — 이상치(불가능한 값) 제거
**배경**: 거리 0마일, 소요시간 0분, 승객 0명인 운행은 **현실에서 있을 수 없는 기록**입니다. 미터기 오작동이나 취소된 호출로 보이며, 그대로 두면 회귀의 기울기를 끌어당깁니다.

**요구사항**:
- `distance > 0` **그리고** `소요시간_분 > 0` **그리고** `passengers > 0` 인 행만 남겨 `df` 에 다시 담으세요.
- 남은 행 수를 `rows_clean`, **제거된 행 수**를 `removed_outlier` 에 담으세요.
- 두 값과 함께, 정제 후 `total`(총액)의 **평균**을 `print` 로 출력하세요.

**예시**
```
rows_clean      → 6220
removed_outlier → 121
```

<details><summary>힌트</summary>

```text
접근방법:
- 세 조건을 모두 만족하는 행만 남긴다. pandas 에서 조건 여러 개는 & 로 잇고 각 조건을 괄호로 감싼다.

세부구현:
1. 제거 전 행 수를 미리 담아 둔다
2. 세 조건(거리·소요시간·승객이 모두 0보다 큼)을 & 로 이어 걸러 df 에 다시 담는다
3. 남은 행 수와 제거된 행 수를 각각 rows_clean, removed_outlier 에 담아 출력한다
```

</details>

In [ ]:
before = df.shape[0]
# 거리 0 · 시간 0 · 승객 0 은 실제 운행일 수 없다 — 통계를 내기 전에 걸러야 평균이 왜곡되지 않는다.
#  조건마다 괄호를 씌워야 한다(& 가 비교보다 먼저 계산돼 오류가 난다).
df = df[(df["distance"] > 0) & (df["소요시간_분"] > 0) & (df["passengers"] > 0)]
rows_clean = df.shape[0]
removed_outlier = before - rows_clean
print("이상치 제거 후 행 수:", rows_clean)
print("제거된 행 수        :", removed_outlier)
print(f'정제 후 총액 평균   = {df["total"].mean():.4f} 달러')

In [ ]:
# [자가채점]
assert rows_clean == 6220, '세 조건(거리·소요시간·승객 > 0)을 모두 걸었는지 확인하세요'
assert removed_outlier == 121
assert (df['distance'] > 0).all() and (df['passengers'] > 0).all()
print("✅ 5단계 통과!")

### 해설 — 문제 1 · 5단계
- **접근법**: **'통계적으로 극단적인 값'과 '물리적으로 불가능한 값'은 다릅니다.** 여기서 지운 것은 후자라 판단이 명확합니다(거리 0·승객 0인 운행은 존재할 수 없음). 반면 '요금이 유난히 비싼 운행'은 공항 장거리일 수 있어 함부로 지우면 안 됩니다.
- **흔한 실수**: 조건을 `and` 로 잇거나 괄호를 빼면 오류가 납니다. pandas 조건 결합은 `&` 이고 **각 조건을 괄호로** 감쌉니다.
- **더 생각해 볼 점**: 총 213행(92 + 121, 3.3%)을 잃었습니다. 이 정도 손실은 무난하지만, **얼마나 버렸는지 항상 기록**해 두어야 나중에 결과를 의심할 때 되짚을 수 있습니다.

### 6단계 — 정제본 저장
**배경**: 전처리 결과를 파일로 남겨 두면 **뒤 분석은 정제본만 읽어** 바로 시작할 수 있습니다. 실무에서도 '원본은 절대 덮어쓰지 않고, 정제본을 따로 저장' 하는 것이 기본입니다.

**요구사항**:
- 정제된 `df` 를 `output/taxis_정제.csv` 로 저장하세요(**인덱스는 저장하지 않습니다** — `index=False`).
- 저장한 파일을 다시 읽어 `check` 에 담고, 그 **행 수**를 `saved_rows`, **열 수**를 `saved_cols` 에 담으세요.
- 최종 크기를 `print` 로 출력하세요.

**예시**
```
saved_rows → 6220
saved_cols → 16
```

<details><summary>힌트</summary>

```text
접근방법:
- DataFrame 을 CSV 로 내보내는 메서드에 index=False 를 준다.
- 저장이 제대로 됐는지 확인하려면 그 파일을 다시 읽어 크기를 본다.

세부구현:
1. df 를 output/taxis_정제.csv 로 저장한다(index=False)
2. 같은 경로를 read_csv 로 다시 읽어 check 에 담는다
3. check.shape 에서 행 수·열 수를 꺼내 saved_rows, saved_cols 에 담고 출력한다
```

</details>

In [ ]:
df.to_csv("output/taxis_정제.csv", index=False)
# 저장한 파일을 곧바로 다시 읽어 확인한다 — 저장됐다고 믿지 말고 눈으로 본다.
check = pd.read_csv("output/taxis_정제.csv")
saved_rows, saved_cols = check.shape
print("저장 완료 —", saved_rows, "행 ×", saved_cols, "열")
print("\n[정제본 앞 3행]"); display(check.head(3))

In [ ]:
# [자가채점]
assert saved_rows == 6220
assert saved_cols == 16
assert os.path.exists('output/taxis_정제.csv'), '정제본을 저장했는지 확인하세요'
print("✅ 6단계 통과! — 전처리 완료, 이제 분석을 시작할 수 있습니다")

### 해설 — 문제 1 · 6단계
- **접근법**: 전처리와 분석을 **파일로 끊어 두면** 분석을 다시 돌릴 때마다 전처리를 반복하지 않아도 되고, '어떤 데이터로 낸 결과인가'가 파일 하나로 분명해집니다.
- **흔한 실수**: `index=False` 를 빼면 저장할 때마다 `Unnamed: 0` 열이 하나씩 늘어나 열 수가 어긋납니다.
- **더 생각해 볼 점**: 실무에서는 여기에 **정제 규칙을 기록한 문서**(무엇을 왜 버렸는지)를 함께 남깁니다. 숫자만 남기고 규칙을 잃으면, 몇 달 뒤 같은 분석을 재현할 수 없습니다.

---
## 2. 어떤 집단이 다른가 — 가정 점검과 비교 검정

**배경**: 정제본으로 세 가지 질문에 답합니다. **1) 결제수단에 따라 팁을 더 후하게 주는가?** **2) 승차 자치구에 따라 팁비율이 다른가?** **3) 택시 종류와 자치구는 서로 관련이 있는가?** 검정을 고르기 전에 **가정부터 점검**하고, 결과가 나오면 **그 숫자가 어떻게 만들어졌는지** 확인합니다.

> ⚠️ **문제 1 을 먼저 끝내세요.** 이 문제는 문제 1 이 저장한 정제본을 **새로 읽어** 시작합니다.

| 단계 | 확인 항목 |
|---|---|
| 1단계 | 정제본 로드 — **6220** 행, 결제수단별 건수 |
| 2단계 | 정규성(카드 팁비율) · 등분산 점검 → 검정 선택 |
| 3단계 | 결제수단별 팁비율 — 검정통계량 **7469831.0**, CLES **0.9506** |
| 4단계 | **결과 뒤집어 보기** — 현금 결제의 팁비율 0 비율 **1.0** |
| 5단계 | 카드 결제만으로 자치구별 팁비율 — H **544.18**, Dunn 사후검정 유의한 쌍 **6/6** |
| 6단계 | 택시 종류 × 승차 자치구 카이제곱 — Cramér's V **0.621**, 조정된 잔차 |잔차|>2 **8칸** |

### 1단계 — 정제본 불러오기
**요구사항**:
- 문제 1 에서 저장한 `output/taxis_정제.csv` 를 `taxi` 에 불러오세요(**문제 1 의 `df` 를 이어 쓰지 말고 새로 읽습니다** — 문제 간 오염 방지).
- 행 수를 `n_clean` 에 담으세요.

> `FileNotFoundError` 가 난다면 **문제 1 을 끝까지 실행하지 않은 것**입니다 — 문제 1 의 6단계까지 마쳐 정제본이 저장돼 있어야 합니다.
- 결제수단(`payment`)별 건수를 `pay_counts`(Series)에 담고 출력하세요.

**예시**
```
n_clean                    → 6220
pay_counts['credit card']  → 4457
pay_counts['cash']         → 1763
```

<details><summary>힌트</summary>

```text
접근방법:
- 정제본 경로를 read_csv 로 읽는다.
- 범주별 건수는 그 열의 값 세기 메서드로 구한다.

세부구현:
1. output/taxis_정제.csv 를 taxi 에 담는다
2. taxi.shape 의 행 수를 n_clean 에 담는다
3. payment 열의 값별 개수를 pay_counts 에 담아 출력한다
```

</details>

In [ ]:
# 원본이 아니라 1부에서 만든 정제본을 읽는다 — 검정은 걸러진 데이터 위에서만 뜻이 있다.
taxi = pd.read_csv("output/taxis_정제.csv")
n_clean = taxi.shape[0]
pay_counts = taxi["payment"].value_counts()
print("정제본 행 수:", n_clean)
print("\n[결제수단별 건수]"); print(pay_counts.to_string())

In [ ]:
# [자가채점]
assert n_clean == 6220, '문제 1 의 정제본(6220행)을 새로 읽었는지 확인하세요'
assert pay_counts['credit card'] == 4457
assert pay_counts['cash'] == 1763
print("✅ 1단계 통과!")

### 2단계 — 가정 점검: 정규성과 등분산
**배경**: 두 집단의 팁비율을 비교하기 전에, **평균을 쓰는 t-검정을 써도 되는지** 확인합니다. 정규성은 `pg.normality`, 등분산은 `pg.homoscedasticity`(Levene)로 봅니다.

**요구사항**:
- **카드 결제(`payment == 'credit card'`)** 행의 `팁비율` 을 `rate_card` 에 담으세요.
- `pg.normality` 로 `rate_card` 의 정규성을 검정해 **W**(`['W'].iloc[0]`)를 `norm_w`, **pval**(`['pval'].iloc[0]`)을 `norm_p` 에 담으세요.
- `pg.homoscedasticity(data=taxi, dv='팁비율', group='payment')` 로 등분산을 검정해 **pval**(`['pval'].iloc[0]`)을 `levene_p` 에 담으세요.
- **`rate_card` 의 Q-Q Plot 도 함께** 그리세요(`pg.qqplot(rate_card, dist='norm', ax=ax)`, 자가채점 없음). 검정은 '기각' 이라는 결론만 주지만, 그림은 **어떻게 벗어났는지**(치우침·꼬리)를 보여 줍니다.
- 세 값을 출력하고, **정규성·등분산이 모두 깨졌다는 것**을 확인하세요.
- 검정통계량은 소수 **셋째 자리**까지 비교합니다.

> **왜 카드만 검정하나요?** 현금 결제 쪽은 값이 전부 똑같아서 정규성 검정이 아예 계산되지 않습니다(`W` 가 `nan`). **그 자체가 데이터에 뭔가 있다는 신호**인데, 무엇인지는 4단계에서 밝힙니다.

**예시**
```
round(norm_w, 3)  → 0.939
norm_p            → 1.6e-39   (0.05 보다 훨씬 작음 → 정규성 기각)
levene_p          → 0.0       (0.05 보다 작음 → 등분산도 기각)
```

<details><summary>힌트</summary>

```text
접근방법:
- 결제수단으로 행을 걸러 팁비율 열만 꺼낸다.
- 정규성은 그 시리즈에, 등분산은 긴 형태(dv=팁비율, group=payment)로 점검한다.

세부구현:
1. payment 가 'credit card' 인 행의 팁비율을 rate_card 에 담는다
2. pg.normality(rate_card) 결과의 ['W']·['pval'] 첫 값을 norm_w, norm_p 에 담는다
3. pg.homoscedasticity 로 등분산 pval 을 levene_p 에 담는다
4. plt.subplots 로 축을 만들고 pg.qqplot 으로 rate_card 의 Q-Q Plot 을 그린다
4. 세 값을 출력하고 0.05 와 비교해 해석한다
```

</details>

In [ ]:
# 검정을 고르기 전에 가정부터 확인한다 — 정규성(집단마다)과 등분산(집단 사이) 둘 다.
rate_card = taxi[taxi["payment"] == "credit card"]["팁비율"]
nt = pg.normality(rate_card)
display(nt)
fig, ax = plt.subplots(figsize=(6, 5))
pg.qqplot(rate_card, dist='norm', ax=ax)
ax.set_title("카드 결제 팁비율 Q-Q Plot")
plt.show()
print("가운데는 기준선을 따르지만 양 끝이 크게 벗어난다 — 정규가 아니다.")
norm_w = nt["W"].iloc[0]
norm_p = nt["pval"].iloc[0]
lv = pg.homoscedasticity(data=taxi, dv="팁비율", group="payment")
display(lv)
levene_p = lv["pval"].iloc[0]
norm_verdict = "정규" if norm_p >= 0.05 else "정규성 기각"
lev_verdict = "등분산" if levene_p >= 0.05 else "등분산 기각"
print(f'카드 팁비율 정규성 W = {norm_w:.3f}, p = {norm_p:.3g} -> {norm_verdict}')
print(f'등분산(Levene)  p = {levene_p:.3g} -> {lev_verdict}')
print("\n두 가정이 모두 깨졌다 -> 평균 대신 순위를 쓰는 비모수 검정으로 간다.")

In [ ]:
# [자가채점]
assert abs(norm_w - 0.939) < 0.01
assert norm_p < 0.05, '정규성이 기각되는 것이 정상입니다'
assert levene_p < 0.05
print("✅ 2단계 통과!")

### 해설 — 문제 2 · 2단계
- **접근법**: 정규성 p ≈ 1.6e−39, 등분산 p ≈ 0 으로 **두 가정이 모두 기각**됩니다. 그래서 평균을 쓰는 t-검정 대신 **순위 기반 Mann-Whitney U**(`pg.mwu`)로 갑니다. 검정은 결과를 보고 고르는 것이 아니라 **가정 점검 결과로** 고르는 것입니다.
- **흔한 실수**: 표본이 6,220건으로 크면 정규성 검정은 **아주 작은 이탈에도 기각**됩니다. 그래서 p 만 보지 말고 히스토그램·Q-Q 로 **얼마나** 벗어났는지 함께 봐야 합니다. 여기서는 형태 자체가 크게 치우쳐 비모수가 맞습니다.
- **더 생각해 볼 점**: 현금 쪽 정규성 검정이 `nan` 을 돌려주는 것은 **그 집단의 값이 전부 동일**해 분산이 0이기 때문입니다. 검정이 실패하는 것도 데이터에 대한 정보입니다 — 그냥 넘기지 말고 원인을 찾아야 합니다.

### 3단계 — 결제수단별 팁비율 비교 (검정 선택)
**배경**: 2단계에서 정규성·등분산이 모두 깨진 것을 확인했습니다. 그 결과에 맞는 검정을 **스스로 골라** '카드 결제와 현금 결제의 팁비율이 다른가'를 봅니다.

**요구사항**:
- 현금 결제(`payment == 'cash'`) 행의 `팁비율` 을 `rate_cash` 에 담으세요.
- 2단계 결과를 근거로 **적절한 검정을 골라** 실행하고 결과 표를 `display` 하세요. 두 집단은 서로 다른 사람이므로 **독립 2표본**이고, 정규성이 깨졌으니 **순위 기반**으로 갑니다. 인자는 **카드(`rate_card`)를 첫 번째, 현금(`rate_cash`)을 두 번째**로 넣으세요(순서가 바뀌면 값이 달라집니다).
- 결과 표에서 **U_val** 을 `u_stat`, **p_val** 을 `mwu_p`, 효과크기 **CLES** 를 `cles` 에 담으세요.
- **[단측 검정]** 실무 가설은 방향이 있는 경우가 많습니다 — "**카드 결제가 현금보다 팁비율이 높다**" 처럼요. `alternative='greater'` 로 단측 검정을 해 p 를 `p_greater` 에, 방향을 **반대로 잡은** `alternative='less'` 의 p 를 `p_less` 에 담고 둘을 나란히 출력하세요.
- 세 값을 출력하세요. `cles` 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
u_stat          → 7469831.0
mwu_p           → 0.0        (사실상 0)
round(cles, 3)  → 0.951      # 카드 쪽이 더 클 확률이 95%
p_greater       → 0.0        # 방향을 맞게 잡음
p_less          → 1.0        # 방향을 반대로 잡으면 결론이 완전히 뒤집힌다
```
> **CLES(공통언어 효과크기)** 는 '앞 집단에서 하나, 뒤 집단에서 하나를 뽑았을 때 앞이 더 클 확률'입니다. 0.5 면 차이 없음, 1.0 에 가까울수록 앞 집단이 압도적으로 큽니다.

<details><summary>힌트</summary>

```text
접근방법:
- 정규성이 깨진 독립 2집단 비교 = 순위 기반 비모수 검정. 교안 4절에서 다룬 그 함수다.
- 결과 표에서 검정통계량·p-value·CLES 열을 꺼낸다.

세부구현:
1. payment 가 'cash' 인 행의 팁비율을 rate_cash 에 담는다
2. 그 검정 함수를 (rate_card, rate_cash) 순서로 호출해 결과 표를 받아 display 한다
3. 결과 표의 U_val·p_val·CLES 첫 값을 u_stat, mwu_p, cles 에 담아 출력한다
4. 같은 호출에 alternative 를 greater / less 로 각각 주어 단측 p 두 개를 구해 비교한다
```

</details>

In [ ]:
rate_cash = taxi[taxi["payment"] == "cash"]["팁비율"]
mw = pg.mwu(rate_card, rate_cash)
display(mw)
u_stat = mw["U_val"].iloc[0]
mwu_p = mw["p_val"].iloc[0]
cles = mw["CLES"].iloc[0]
print(f'U 통계량 = {u_stat:.1f}')
print(f'p-value  = {mwu_p:.3g} -> {"유의" if mwu_p < 0.05 else "유의하지 않음"}')
print(f'CLES     = {cles:.3f}  (카드 쪽 팁비율이 더 클 확률)')

# 단측 검정 — '카드가 현금보다 높다'는 방향 가설 vs 반대로 잡았을 때
p_greater = pg.mwu(rate_card, rate_cash, alternative="greater")["p_val"].iloc[0]
p_less = pg.mwu(rate_card, rate_cash, alternative="less")["p_val"].iloc[0]
print(f'\n[단측] 카드 > 현금 : p = {p_greater:.3g}  (방향을 맞게 잡음)')
print(f'[단측] 카드 < 현금 : p = {p_less:.3g}  (방향을 반대로 잡으면 결론이 뒤집힌다)')
print("방향은 데이터를 보기 전에 정해야 한다 — 결과를 보고 유리한 쪽으로 바꾸면 1종 오류가 부푼다.")

In [ ]:
# [자가채점]
assert abs(u_stat - 7469831.0) < 1, '카드(rate_card)를 첫 인자로 넣었는지 확인하세요'
assert mwu_p < 0.001
assert abs(cles - 0.951) < 0.01
assert p_greater < 0.001, '카드가 더 높다는 방향이므로 greater 의 p 가 작아야 합니다'
assert p_less > 0.99, '방향을 반대로 잡으면 p 가 1 에 가까워집니다'
print("✅ 3단계 통과!")

### 4단계 — 결과를 뒤집어 보기: 이 차이는 진짜인가
**배경**: p 는 사실상 0, CLES 는 0.95 로 **엄청난 차이**가 나왔습니다. 여기서 "현금 손님은 팁에 인색하다" 고 결론 내리면 **틀립니다.** 결론을 내기 전에 **그 숫자가 어떻게 만들어졌는지** 반드시 확인하세요.

**요구사항**:
- 결제수단별 `팁비율` 의 **평균·중앙값**을 구해 출력하세요.
- 현금 결제 중 **팁비율이 정확히 0인 행의 비율**을 `cash_zero_ratio` 에, 카드 결제 중 같은 비율을 `card_zero_ratio` 에 담으세요.
- 두 비율을 출력하고, **무엇이 이상한지** 눈으로 확인하세요.

**예시**
```
cash_zero_ratio → 1.0      # 현금 결제는 100% 가 팁 0
round(card_zero_ratio, 3) → 0.099
```

<details><summary>힌트</summary>

```text
접근방법:
- 결제수단으로 묶어 팁비율의 평균·중앙값을 함께 구한다.
- '값이 0인 비율' 은 (시리즈 == 0) 의 평균으로 구할 수 있다(True=1, False=0).

세부구현:
1. payment 로 groupby 해 팁비율의 mean·median 을 구해 출력한다
2. rate_cash 에서 (값 == 0) 의 평균을 cash_zero_ratio 에 담는다
3. rate_card 에서 같은 값을 card_zero_ratio 에 담아 함께 출력한다
```

</details>

In [ ]:
summary = taxi.groupby("payment")["팁비율"].agg(["count", "mean", "median"]).round(4)
display(summary)
# 참/거짓의 평균은 곧 '참인 비율'이다 — 여기서는 팁비율이 정확히 0 인 운행의 비율.
#  검정이 유의했다고 끝내지 말고, 그 차이가 어디서 왔는지 이렇게 파고들어야 한다.
cash_zero_ratio = (rate_cash == 0).mean()
card_zero_ratio = (rate_card == 0).mean()
print(f'현금 결제 중 팁비율 0 인 비율 = {cash_zero_ratio:.4f}  <- 100%')
print(f'카드 결제 중 팁비율 0 인 비율 = {card_zero_ratio:.4f}')
print()
print("현금 결제는 단 한 건도 팁이 기록되지 않았다.")
print("승객이 인색한 것이 아니라, 현금 팁은 미터기에 입력되지 않아 0 으로 남은 것이다.")
print("즉 이 검정은 승객 행동이 아니라 '기록 방식'의 차이를 잡아낸 것이다.")

In [ ]:
# [자가채점]
assert abs(cash_zero_ratio - 1.0) < 1e-9, '현금 결제의 팁비율이 0 인 비율입니다 — 정확히 1.0 이 나옵니다'
assert abs(card_zero_ratio - 0.099) < 0.01
print("✅ 4단계 통과! — 검정 결과를 해석하기 전에 데이터가 어떻게 기록됐는지 확인해야 합니다")

### 해설 — 문제 2 · 4단계
- **접근법**: 현금 결제 1,763건이 **전부 팁 0** 입니다. 뉴욕 택시 미터기는 카드 결제 팁만 자동 기록하고 **현금 팁은 시스템에 남지 않습니다.** 그래서 '결제수단별 팁비율' 검정은 통계적으로 완벽히 유의하지만, **측정하려던 것(승객의 팁 문화)을 측정하지 못했습니다.**
- **흔한 실수**: p 가 0에 가깝고 효과크기까지 크면 '확실한 발견'이라고 믿기 쉽습니다. 하지만 **p 는 데이터가 어떻게 수집됐는지 알려 주지 않습니다.** 한 집단의 값이 전부 동일하다는 것은 거의 항상 행동이 아니라 **기록·수집 과정**의 문제입니다.
- **더 생각해 볼 점**: 이런 것을 **측정 편향**이라 합니다. 올바른 대응은 두 가지입니다 — 1) 현금 결제를 분석에서 제외하고 **카드 결제 안에서만** 팁을 비교하거나(5단계에서 이렇게 합니다), 2) 팁을 아예 분석 대상에서 빼는 것입니다.

### 5단계 — 카드 결제만으로 자치구별 팁비율 비교 (검정 선택 + 그래프)
**배경**: 4단계에서 현금 데이터를 믿을 수 없다는 것을 알았으니, **카드 결제만 남겨** '승차 자치구에 따라 팁비율이 다른가'를 봅니다. 이번엔 **자치구가 4개**라 3단계와 상황이 다릅니다 — **3집단 이상**이고 정규성은 여전히 깨져 있습니다. 여기에 맞는 검정을 **스스로 고르세요**.

**요구사항**:
- 카드 결제 행만 남긴 데이터프레임을 `card` 에 담으세요.
- 3집단 이상을 순위로 비교하는 검정을 골라 `data=card`, `dv='팁비율'`, `between='pickup_borough'` 로 실행하고 결과 표를 `display` 하세요.
- 결과 표에서 **H** 를 `h_stat`, **p_unc** 를 `kruskal_p` 에 담으세요.
- 자치구별 팁비율 **중앙값**을 `borough_median`(Series)에 담고 출력하세요.
- **사후검정**: 이 검정도 "적어도 한 쌍이 다르다" 까지만 말합니다. **어느 자치구 쌍이 다른지**를 비모수 사후검정 **Dunn**(`sp.posthoc_dunn`, `p_adjust='holm'`)으로 확인해 `dunn_bor` 에 담고 `display` 하세요. **유의한 쌍의 개수**(p < 0.05)를 `n_sig_bor` 에 담으세요 — 대각선을 빼고 세야 하므로 개수를 2로 나눕니다.
- `h_stat` 은 소수 **둘째 자리**까지 비교하고, `kruskal_p` 는 0 에 아주 가까워 `< 1e-100` 인지로 채점됩니다.
- 마지막으로 **자치구별 팁비율 상자그림**(`sns.boxplot`)을 그리세요.

**예시**
```
round(h_stat, 2)               → 544.18
borough_median['Manhattan']    → 0.266
n_sig_bor                      → 6        # 네 자치구의 6쌍이 모두 유의
borough_median['Bronx']        → 0.0      # 카드인데도 중앙값이 0
```
> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day09_가설검정_회귀분석/images/과제/lv3_q2_s5.png" width="620"/>

<details><summary>힌트</summary>

```text
접근방법:
- 결제수단이 카드인 행만 남긴다.
- 3집단 이상의 비모수 비교 = ANOVA 의 비모수 짝. 교안 5절과 LV2 문제 5 에서 쓴 그 함수다.
- 상자그림은 x 에 자치구, y 에 팁비율을 준다.

세부구현:
1. payment 가 'credit card' 인 행만 남겨 card 에 담는다
2. 비모수 3집단 검정 함수를 data=card, dv='팁비율', between='pickup_borough' 로 호출해 표를 받는다
3. 그 표의 H·p_unc 첫 값을 h_stat, kruskal_p 에 담는다
4. pickup_borough 로 묶어 팁비율의 중앙값을 borough_median 에 담아 출력한다
5. sp.posthoc_dunn 에 데이터·val_col·group_col·p_adjust 를 넘겨 쌍별 p 표를 만든다
6. p<0.05 인 칸 수에서 대각선을 빼고 2로 나눠 유의한 쌍 수를 센다
5. 새 figure 를 열고 상자그림을 그린 뒤 제목·축 라벨을 달고 보여 준다
```

</details>

In [ ]:
card = taxi[taxi["payment"] == "credit card"]
kr = pg.kruskal(data=card, dv="팁비율", between="pickup_borough")
display(kr)
h_stat = kr["H"].iloc[0]
kruskal_p = kr["p_unc"].iloc[0]
borough_median = card.groupby("pickup_borough")["팁비율"].median()
print(f'H 통계량 = {h_stat:.2f}')
print(f'p-value  = {kruskal_p:.3g}')
print("\n[자치구별 팁비율 중앙값 — 카드 결제만]"); print(borough_median.round(4).to_string())

# 사후검정 — 어느 자치구 쌍이 다른가 (Dunn, Holm 보정)
dunn_bor = sp.posthoc_dunn(card, val_col="팁비율", group_col="pickup_borough", p_adjust="holm")
display(dunn_bor.round(6))
n_sig_bor = int(((dunn_bor < 0.05).sum().sum()) / 2)
print("유의한 쌍 =", n_sig_bor, "/ 6 — 네 자치구가 서로 모두 다르다")

plt.figure(figsize=(7, 5))
ax = sns.boxplot(data=card, x="pickup_borough", y="팁비율")
ax.set_title("승차 자치구별 팁비율 (카드 결제만)")
ax.set_xlabel("승차 자치구"); ax.set_ylabel("팁비율")
plt.show()

In [ ]:
# [자가채점]
assert abs(h_stat - 544.18) < 0.1, '카드 결제만 남긴 뒤, 3집단 이상용 비모수 검정을 썼는지 확인하세요'
assert n_sig_bor == 6, 'Dunn 사후검정에서 6쌍이 모두 유의합니다'
assert kruskal_p < 1e-100
assert abs(borough_median['Manhattan'] - 0.266) < 0.01
assert abs(borough_median['Bronx'] - 0.0) < 0.01
print("✅ 5단계 통과!")

### 해설 — 문제 2 · 5단계
- **접근법**: 카드 결제 안에서도 자치구별 차이가 뚜렷합니다(H = 544.18, p ≈ 1.3e−117). 중앙값을 보면 **Manhattan 0.266 · Queens 0.212 인데 Bronx·Brooklyn 은 0** 입니다.
- **흔한 실수**: 카드 결제만 남기지 않고 전체로 돌리면 4단계에서 본 **현금 0** 이 자치구마다 다른 비율로 섞여 들어가, '자치구 차이'가 아니라 '자치구별 현금 결제 비율 차이'를 재게 됩니다.
- **더 생각해 볼 점**: Bronx·Brooklyn 의 중앙값이 0인 것은 또 다른 질문을 낳습니다 — 이 지역은 `green`(외곽 전용) 택시 비중이 높고 길거리에서 손을 들어 잡는 방식이라 팁 문화가 다를 수 있습니다. **6단계에서 택시 종류와 자치구의 관계**를 확인합니다.

### 6단계 — 택시 종류와 자치구는 관련이 있는가 (카이제곱)
**배경**: 5단계 끝의 의문 — 'Bronx·Brooklyn 은 다른 종류의 택시가 다니는 것 아닐까?' 를 확인합니다. **택시 종류(`color`)와 승차 자치구(`pickup_borough`)** 는 둘 다 범주형이므로 **카이제곱 독립성 검정**으로 관련성을 보고, 세기는 **Cramér's V** 로 잽니다.

**요구사항**:
- `pg.chi2_independence(data=taxi, x='color', y='pickup_borough')` 를 실행하세요. 이 함수는 **(기대빈도표, 관측빈도표, 통계량표)** 세 개를 순서대로 돌려줍니다. 관측 교차표를 `display` 하세요.
- 통계량표에서 **표준 Pearson 검정 행**(`test == 'pearson'`)을 골라 `chi2`(→`chi2`), `pval`(→`chi_p`), `cramer`(→`cramers_v`) 를 꺼내세요.
- **사후분석**: 카이제곱은 "관련이 있다" 까지만 말합니다. **어느 칸이 기대보다 두드러지는지**는 **조정된 잔차**로 봅니다 — `Table(observed.values).standardized_resids` 로 계산해 `observed` 와 같은 행·열 이름을 붙인 DataFrame `resid_taxi` 에 담고 `display` 하세요.
- **|조정된 잔차| > 2 인 칸의 개수**를 `n_strong` 에 담으세요. 양수면 기대보다 **많다**, 음수면 **적다**는 뜻입니다.
- 세 값을 출력하세요. 검정통계량·Cramér's V 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(chi2, 3)      → 2401.338
chi_p               → 0.0
round(cramers_v, 3) → 0.621     # 0.5 이상이면 매우 강한 연관
n_strong            → 8         # 8칸 모두 |잔차| > 2
```

<details><summary>힌트</summary>

```text
접근방법:
- 두 범주형 변수의 관련성은 카이제곱 독립성 검정으로 본다 — (기대, 관측, 통계량) 세 표를 준다.
- 통계량표의 test=='pearson' 행이 표준 카이제곱이고, cramer 열이 효과크기다.

세부구현:
1. 지문의 함수를 호출해 세 반환값을 expected, observed, chi_stats 로 한 번에 받는다
2. observed 를 display 해 교차표를 눈으로 본다
3. chi_stats 에서 test 가 'pearson' 인 행을 골라 chi2·pval·cramer 의 첫 값을 담는다
4. Table(관측표.values).standardized_resids 로 조정된 잔차를 구해 행·열 이름을 붙여 resid_taxi 에 담는다
5. 절댓값이 2를 넘는 칸 수를 세어 n_strong 에 담는다
```

</details>

In [ ]:
expected, observed, chi_stats = pg.chi2_independence(data=taxi, x="color", y="pickup_borough")
print("[관측 교차표] 택시 종류 × 승차 자치구"); display(observed)
pearson = chi_stats[chi_stats["test"] == "pearson"]
chi2 = pearson["chi2"].iloc[0]
chi_p = pearson["pval"].iloc[0]
cramers_v = pearson["cramer"].iloc[0]
print(f'카이제곱 = {chi2:.3f}')
print(f'p-value  = {chi_p:.3g} -> {"독립이 아니다(관련 있음)" if chi_p < 0.05 else "독립"}')
print(f"Cramér's V = {cramers_v:.3f}  (0.1 약함 / 0.3 중간 / 0.5 이상 매우 강함)")
print()
print("Manhattan 은 노란 택시가 거의 전부인데, Bronx 는 초록 택시가 다수다.")

# 사후분석 — 어느 칸이 기대보다 두드러지나 (조정된 잔차)
resid_taxi = pd.DataFrame(Table(observed.values).standardized_resids,
                          index=observed.index, columns=observed.columns)
display(resid_taxi.round(2))
n_strong = int((resid_taxi.abs() > 2).sum().sum())
print("|조정된 잔차| > 2 인 칸 =", n_strong, "개")
print("노란 택시 x Manhattan 이 +45.9 로 가장 크다 — 기대보다 압도적으로 많다는 뜻이다.")
print()
print("자치구별 팁비율 차이에는 '어떤 택시가 다니는가'가 섞여 있다 — 자치구만의 효과로 볼 수 없다.")

In [ ]:
# [자가채점]
assert abs(chi2 - 2401.338) < 0.01
assert chi_p < 0.001
assert abs(cramers_v - 0.621) < 0.01, "통계량표 pearson 행의 cramer 열이 Cramér's V 예요"
assert n_strong == 8, '2x4 표의 여덟 칸 모두 |조정된 잔차| 가 2 를 넘습니다'
assert abs(resid_taxi.loc['yellow', 'Manhattan'] - 45.94) < 0.1
print("✅ 6단계 통과!")

**해석 (모범 서술)**

3단계의 Mann-Whitney 결과는 p ≈ 0, CLES = 0.951 로 통계적으로 완벽하게 유의하지만, **결론으로 쓸 수 없다.** 4단계에서 확인했듯 현금 결제 1,763건은 **100% 가 팁 0** 인데, 이는 승객이 팁을 주지 않아서가 아니라 **현금 팁이 미터기에 기록되지 않기 때문**이다. 검정은 승객 행동이 아니라 **기록 방식의 차이**를 잡아냈다. p-value 는 데이터가 어떻게 수집됐는지 알려 주지 않으므로, 유의성만 보고 해석하면 완전히 틀린 결론에 이른다.

그래서 5단계에서는 **팁이 실제로 기록되는 카드 결제만 남겨** 비교했다. 그 안에서도 자치구별 차이는 뚜렷했다(H = 544.18, p ≈ 1.3e−117). 중앙값은 Manhattan 0.266 · Queens 0.212 인 반면 Bronx·Brooklyn 은 0 이었다.

다만 **'자치구가 팁비율의 원인'이라고 말할 수는 없다.** 6단계의 카이제곱에서 택시 종류와 자치구는 매우 강하게 얽혀 있었다(Cramér's V = 0.621) — Manhattan 은 노란 택시가 거의 전부이고 Bronx 는 초록 택시가 다수다. 따라서 자치구별 팁비율 차이에는 **택시 종류·호출 방식·승객 구성** 같은 요인이 섞여 있으며, 이것은 관찰 데이터이므로 **상관이지 인과가 아니다.**

---
## 3. 요금은 무엇으로 정해지는가 — 회귀와 의사결정

**배경**: 마지막 질문입니다. **택시 총액(`total`)은 무엇으로 설명되는가?** 거리 하나로 시작해 소요시간·승객 수를 더해 가며, 각 변수의 기여를 계수와 p-value 로 읽고 **이 회귀를 믿어도 되는지** 진단까지 합니다.

> ⚠️ 이 문제도 정제본을 **새로 읽어** 시작합니다.

| 단계 | 확인 항목 |
|---|---|
| 1단계 | 정제본 로드 — **6220** 행 |
| 2단계 | 단순회귀 `total ~ distance` — 기울기 **3.242**, R² **0.882** |
| 3단계 | 다중회귀 `+ 소요시간_분 + passengers` — Adj R² **0.906** |
| 4단계 | 잔차 진단 그래프 — Durbin-Watson **1.781** |
| 5단계 | 예측과 의사결정 서술 |

### 1단계 — 정제본 다시 불러오기
**요구사항**:
- `output/taxis_정제.csv` 를 `tx` 에 불러오세요(문제 2 의 `taxi` 를 이어 쓰지 말고 **새로 읽습니다**).
- 행 수를 `n_reg` 에 담고, `total`(총액)의 **평균**을 `total_mean` 에 담아 출력하세요.

> 여기서도 `FileNotFoundError` 가 나면 문제 1 을 먼저 끝내세요.
- `total_mean` 은 소수 **셋째 자리**까지 비교합니다.

**예시**
```
n_reg                  → 6220
round(total_mean, 3)   → 18.303
```

<details><summary>힌트</summary>

```text
접근방법:
- 문제 1 이 저장한 정제본을 read_csv 로 읽는다.

세부구현:
1. output/taxis_정제.csv 를 tx 에 담는다
2. 행 수를 n_reg, total 열의 평균을 total_mean 에 담아 출력한다
```

</details>

In [ ]:
# 회귀도 같은 정제본에서 출발한다 — 앞 파트와 같은 데이터라야 결과를 이어서 읽을 수 있다.
tx = pd.read_csv("output/taxis_정제.csv")
n_reg = tx.shape[0]
total_mean = tx["total"].mean()
print("행 수      :", n_reg)
print(f'총액 평균  = {total_mean:.3f} 달러')

In [ ]:
# [자가채점]
assert n_reg == 6220
assert abs(total_mean - 18.303) < 0.01
print("✅ 1단계 통과!")

### 2단계 — 단순 선형회귀: 거리로 총액 설명하기
**배경**: 가장 단순한 모형부터 시작합니다. **거리 하나로 총액이 얼마나 설명되는가?**

**요구사항**:
- `smf.ols('total ~ distance', data=tx).fit()` 로 적합해 `m1` 에 담으세요.
- 절편을 `b0`, `distance` 계수를 `b_dist`, 결정계수를 `r2_simple` 에 담으세요.
- 세 값을 출력하고, **거리가 1마일 늘 때 총액이 얼마나 오르는지** 문장으로 함께 출력하세요.
- 계수·R² 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(b0, 3)        → 8.531
round(b_dist, 3)    → 3.242    # 1마일당 약 3.24달러
round(r2_simple, 3) → 0.882
```

<details><summary>힌트</summary>

```text
접근방법:
- ols 에 '종속변수 ~ 독립변수' 식과 데이터를 넣고 fit 한다.
- 적합 결과 객체에서 params 로 계수를, rsquared 로 결정계수를 꺼낸다.

세부구현:
1. total 을 distance 로 설명하는 식으로 모형을 적합해 m1 에 담는다
2. params['Intercept']·params['distance'] 를 b0, b_dist 에 담는다
3. rsquared 를 r2_simple 에 담고 세 값을 해석 문장과 함께 출력한다
```

</details>

In [ ]:
# 먼저 설명변수 하나로 기준선을 만든다 — 뒤에서 변수를 더했을 때 무엇이 나아졌는지 견주기 위해서다.
m1 = smf.ols("total ~ distance", data=tx).fit()
b0 = m1.params["Intercept"]
b_dist = m1.params["distance"]
r2_simple = m1.rsquared
print(f'회귀식: 총액 = {b0:.3f} + {b_dist:.3f} x 거리')
print(f'R^2   = {r2_simple:.3f} -> 거리 하나로 총액 변동의 약 {r2_simple * 100:.0f}% 를 설명')
print(f'해석  : 거리가 1마일 늘면 총액이 평균 {b_dist:.2f} 달러 오른다.')

In [ ]:
# [자가채점]
assert abs(b0 - 8.531) < 0.01
assert abs(b_dist - 3.242) < 0.01
assert abs(r2_simple - 0.882) < 0.01
print("✅ 2단계 통과!")

### 3단계 — 다중 선형회귀: 소요시간과 승객 수를 더하면
**배경**: 택시 요금은 거리뿐 아니라 **막힌 시간**에도 붙습니다(시간 요금). 소요시간과 승객 수를 넣어 설명력이 얼마나 오르는지, 각 변수가 유의한지 봅니다.

**요구사항**:
- `smf.ols('total ~ distance + 소요시간_분 + passengers', data=tx).fit()` 로 적합해 `m2` 에 담으세요.
- `distance` 계수를 `c_dist`, `소요시간_분` 계수를 `c_dur`, `passengers` 계수를 `c_pass` 에 담으세요.
- `passengers` 계수의 p-value 를 `p_pass`, 모형의 **조정 결정계수**를 `adj_r2` 에 담으세요.
- `print(m2.summary())` 로 전체 표를 보고, 위 값들을 함께 출력하세요.
- 계수·조정 R² 는 소수 **셋째 자리**, p값은 소수 **넷째 자리**까지 비교합니다.

**예시**
```
round(c_dist, 3)   → 2.476     # 소요시간을 넣자 3.242 에서 줄었다
round(c_dur, 3)    → 0.302
round(c_pass, 3)   → 0.100
round(p_pass, 4)   → 0.0159    # 유의하긴 하다
round(adj_r2, 3)   → 0.906
```

<details><summary>힌트</summary>

```text
접근방법:
- ols 식에 독립변수를 + 로 이어 세 개를 넣는다.
- 조정 결정계수는 rsquared_adj 로 꺼낸다.

세부구현:
1. total 을 distance·소요시간_분·passengers 로 설명하는 식으로 적합해 m2 에 담는다
2. params 에서 세 계수를, pvalues 에서 passengers 의 p 를 꺼낸다
3. rsquared_adj 를 adj_r2 에 담고 summary 와 함께 출력한다
```

</details>

In [ ]:
# 변수를 더하면 거리 계수가 작아진다 — 거리가 혼자 안고 있던 설명력을 소요시간이 나눠 가졌기 때문이다.
#  '계수가 줄었다 = 나빠졌다'가 아니라 '거리만의 순수한 몫'에 가까워진 것이다.
m2 = smf.ols("total ~ distance + 소요시간_분 + passengers", data=tx).fit()
print(m2.summary())
c_dist = m2.params["distance"]
c_dur = m2.params["소요시간_분"]
c_pass = m2.params["passengers"]
p_pass = m2.pvalues["passengers"]
adj_r2 = m2.rsquared_adj
print()
print(f'거리 계수     = {c_dist:.3f}  (단순회귀의 {b_dist:.3f} 보다 작아졌다)')
print(f'소요시간 계수 = {c_dur:.3f}  (1분당 {c_dur:.2f} 달러)')
print(f'승객수 계수   = {c_pass:.3f}  (p = {p_pass:.4f})')
print(f'조정 R^2      = {adj_r2:.3f}  (단순회귀 {r2_simple:.3f} 에서 상승)')

In [ ]:
# [자가채점]
assert abs(c_dist - 2.476) < 0.01
assert abs(c_dur - 0.302) < 0.01
assert abs(c_pass - 0.100) < 0.01
assert abs(p_pass - 0.0159) < 0.001
assert abs(adj_r2 - 0.906) < 0.01
print("✅ 3단계 통과!")

### 해설 — 문제 3 · 3단계
- **접근법**: 소요시간을 넣자 거리 계수가 **3.242 → 2.476 으로 줄었습니다.** 단순회귀의 거리 계수에는 '거리가 길면 시간도 오래 걸린다'는 효과가 **섞여 있었기** 때문입니다. 다중회귀의 계수는 **다른 변수를 고정한 채**의 순수한 효과라 이렇게 달라집니다.
- **흔한 실수**: `passengers` 의 p = 0.0159 를 보고 '승객 수가 요금에 유의하게 영향을 준다'고 강조하기 쉽습니다. 하지만 계수는 **0.100 달러** — 승객이 한 명 늘어도 10센트입니다. **유의성과 실질적 크기는 다른 질문**입니다. 표본이 6,220건으로 크면 사소한 차이도 유의해집니다.
- **더 생각해 볼 점**: 뉴욕 택시 요금은 실제로 '기본요금 + 거리 요금 + 정차 시간 요금'으로 매겨집니다. 회귀 결과가 **요금 체계와 맞아떨어지는 것**은 좋은 신호입니다. 모형이 도메인 지식과 어긋나면 데이터나 모형을 의심해야 합니다.

### 4단계 — 잔차 진단: 이 회귀를 믿어도 되는가 (그래프)
**배경**: 선형회귀는 **LINE 네 가정** — **선형성**(Linearity)·**독립성**(Independence)·**정규성**(Normality)·**등분산성**(Equal variance) — 위에서 성립합니다. 이를 **잔차**로 점검합니다. **가정은 넷인데 도구는 셋**입니다: 잔차 vs 적합값 한 그림에서 **선형성·등분산성**을 함께 보고, **정규성**은 Q-Q Plot, **독립성**은 Durbin-Watson 으로 확인합니다.

**요구사항**:
- `plt.subplots(1, 2, figsize=(12, 5))` 로 서브플롯 두 개(`ax1`·`ax2`)를 만드세요.
- **왼쪽(`ax1`)**: `sns.scatterplot` 으로 x=적합값(`m2.fittedvalues`), y=잔차(`m2.resid`) **산점도**를 그리고(점이 많으니 `alpha=0.3`), 잔차 0 위치에 빨간 **수평 점선**을 그으세요.
- **오른쪽(`ax2`)**: 잔차의 **Q-Q Plot** 을 `pg.qqplot(m2.resid, dist='norm', ax=ax2)` 로 그리세요.
- 두 Axes 에 각각 제목을 다세요.
- **Durbin-Watson** 을 `durbin_watson(m2.resid)` 로 구해 `dw` 에 담고 출력하세요.
- `dw` 는 소수 **셋째 자리**까지 비교합니다.

**예시**
```
round(dw, 3) → 1.781    # 2 근처면 잔차가 서로 독립
```
> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="../../day09_가설검정_회귀분석/images/과제/lv3_q3_s4.png" width="760"/>

<details><summary>힌트</summary>

```text
접근방법:
- 서브플롯 두 개를 나란히 만들어 왼쪽엔 적합값-잔차 산점도, 오른쪽엔 잔차 Q-Q Plot 을 그린다.
- Durbin-Watson 은 statsmodels 의 durbin_watson 함수로 한 줄에 구한다.

세부구현:
1. subplots(1, 2, ...) 로 ax1, ax2 두 축을 만든다
2. ax1 에 fittedvalues(x)-resid(y) 산점도를 alpha 를 낮춰 그리고 axhline 으로 0 선을 긋는다
3. ax2 에 pg.qqplot(잔차, dist='norm', ax=ax2) 로 Q-Q Plot 을 그린다
4. 두 축에 제목을 달고 보여 준다
5. durbin_watson(잔차) 로 DW 를 구해 출력한다
```

</details>

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(x=m2.fittedvalues, y=m2.resid, alpha=0.3, ax=ax1)
ax1.axhline(0, color="red", linestyle="--")
ax1.set_title("잔차 vs 적합값")
ax1.set_xlabel("적합값(예측 총액)"); ax1.set_ylabel("잔차")

pg.qqplot(m2.resid, dist="norm", ax=ax2)
ax2.set_title("잔차 Q-Q Plot")
plt.tight_layout()
plt.show()

# 잔차가 서로 독립인지 보는 지표 — 2 근처면 좋고, 0 이나 4 에 가까우면 이웃한 잔차가 붙어 다닌다는 뜻이다.
dw = durbin_watson(m2.resid)
print(f'Durbin-Watson = {dw:.3f}  (2 근처면 잔차가 서로 독립)')
print(f'잔차 평균     = {abs(m2.resid.mean()):.2e}  (최소제곱 회귀에서 항상 0 에 가깝다)')

In [ ]:
# [자가채점]
assert abs(dw - 1.781) < 0.01
print("✅ 4단계 통과!")

### 해설 — 문제 3 · 4단계
- **접근법**: Durbin-Watson 1.781 은 2 에 가까워 **독립성은 무난**합니다. 반면 두 그림은 문제를 보여 줍니다 — 잔차 산점도는 오른쪽으로 갈수록 퍼지는 **깔때기 모양(이분산)** 이고, Q-Q Plot 은 양 끝이 직선에서 크게 벗어나 **꼬리가 두껍습니다.**
- **흔한 실수**: R² 가 0.906 으로 높다고 '좋은 모형'이라고 끝내면 안 됩니다. **설명력과 가정 충족은 별개**입니다. 잔차를 보지 않으면 이분산을 놓칩니다.
- **더 생각해 볼 점**: 이분산이 있으면 계수 추정치 자체는 쓸 만하지만 **표준오차와 p-value 를 그대로 믿기 어렵습니다.** 실무 대응은 로버스트 표준오차를 쓰거나 종속변수를 로그로 바꾸는 것인데, 둘 다 이 단원 범위를 넘습니다 — 지금 필요한 것은 **한계를 알아채고 보고서에 적는 것**입니다.

### 5단계 — 예측과 의사결정
**배경**: 마지막으로 모형을 **실제 예측**에 써 보고, 지금까지의 분석을 하나의 결론으로 정리합니다.

**요구사항**:
- 3단계의 `m2` 로 **거리 3.0마일 · 소요시간 15분 · 승객 1명** 인 운행의 총액을 예측해 `pred_total` 에 담으세요.
  (`m2.predict(pd.DataFrame({'distance': [3.0], '소요시간_분': [15.0], 'passengers': [1]}))` 의 첫 값)
- 예측값을 출력하세요. 소수 **셋째 자리**까지 비교합니다.
- 아래 **서술 셀**에 의사결정 리포트를 적으세요.

**예시**
```
round(pred_total, 3) → 18.393
```

<details><summary>힌트</summary>

```text
접근방법:
- 적합된 모형의 predict 에 예측하려는 조건을 담은 DataFrame 을 넘긴다(열 이름은 학습 때와 같아야 한다).

세부구현:
1. distance·소요시간_분·passengers 를 각각 한 값씩 담은 DataFrame 을 만든다
2. m2.predict 에 넘겨 나온 결과의 첫 값을 pred_total 에 담아 출력한다
```

</details>

In [ ]:
new_ride = pd.DataFrame({"distance": [3.0], "소요시간_분": [15.0], "passengers": [1]})
# 예측할 때는 학습에 쓴 열 이름·개수가 정확히 같아야 한다(하나라도 다르면 오류가 난다).
pred_total = m2.predict(new_ride).iloc[0]
print(f'거리 3.0마일 · 소요시간 15분 · 승객 1명 -> 예측 총액 = {pred_total:.3f} 달러')
print(f'(정제본 전체 평균 총액 {total_mean:.3f} 달러와 비슷한 수준)')

In [ ]:
# [자가채점]
assert abs(pred_total - 18.393) < 0.01
print("✅ 5단계 통과! — LV3 완주")

**의사결정 리포트 (모범 서술)**

1) **총액은 거리와 소요시간으로 대부분 설명된다.** 세 변수 모형의 조정 R² 는 0.906 으로, 총액 변동의 약 91% 를 설명한다. 계수를 단위와 함께 읽으면 **거리 1마일당 약 2.48달러, 소요시간 1분당 약 0.30달러** 가 붙는다. 이는 '기본요금 + 거리요금 + 시간요금' 이라는 실제 뉴욕 택시 요금 체계와 잘 맞아떨어진다. 거리만 넣은 단순회귀에서는 거리 계수가 3.24였는데, 소요시간을 함께 넣자 2.48로 줄었다 — 단순회귀의 거리 계수에 **시간의 효과가 섞여 있었기** 때문이다.

2) **`passengers` 는 유의하지만(p = 0.016) 계수가 0.100달러에 불과하다.** 승객이 한 명 더 타도 요금은 10센트 오를 뿐이다. 표본이 6,220건으로 크면 실질적으로 무의미한 차이도 통계적으로 유의해진다. **'유의한가'와 '얼마나 큰가'는 다른 질문**이므로, 보고서에는 p 만 적지 말고 **계수의 크기를 단위와 함께** 적어야 한다. 요금 정책을 논의한다면 승객 수는 실질적 고려 대상이 아니다.

3) **잔차 진단에서 두 가지 한계가 드러났다.** 잔차 vs 적합값 그림이 오른쪽으로 퍼지는 깔때기 모양이라 **등분산 가정이 깨졌고**(요금이 큰 장거리 운행일수록 예측 오차가 크다), Q-Q Plot 의 양 끝이 직선에서 벗어나 **잔차의 꼬리가 두껍다**. Durbin-Watson 은 1.781 로 독립성은 무난하다. 이분산 때문에 계수의 **표준오차와 p-value 는 그대로 믿기 어렵다** — 다만 계수 추정치 자체와 예측의 큰 그림은 여전히 쓸 만하다.

4) **실제 의사결정에 쓸 때는 세 가지를 주의해야 한다.** 첫째, 이 모형은 **2019년 3월 뉴욕**의 6,220건에서 나온 것이라 다른 시기·도시에 그대로 적용할 수 없다. 둘째, 회귀선은 **관측 범위 안에서만** 믿을 수 있어 100마일짜리 운행을 예측에 넣으면 안 된다(외삽). 셋째, 문제 2 에서 확인했듯 **관찰 데이터라 인과가 아니다** — 특히 현금 결제의 팁이 전부 0이었던 것처럼, 숫자가 어떻게 **기록**됐는지 확인하지 않으면 통계적으로 완벽히 유의한 결론도 완전히 틀릴 수 있다.